In [2]:
import pandas as pd
from sklearn.linear_model import LinearRegression

laps = pd.read_csv("../data/processed/laps_2026.csv")
races = laps[laps["session_name"] == "Race"]
g = races[races["location"] == "Melbourne"]

In [4]:
X = pd.get_dummies(g["driver_number"], prefix = "driver", dtype = float)
X["lap_number"] = g["lap_number"]
X["tyre_age"] = g["tyre_age"]
y = g["lap_duration"]

X.head()

,driver_1,driver_3,driver_5,driver_6,driver_10,driver_11,driver_12,driver_14,driver_16,driver_18,driver_23,driver_30,driver_31,driver_41,driver_43,driver_44,driver_55,driver_63,driver_77,driver_87,lap_number,tyre_age
3515,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2,1
3516,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,2,1
3517,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2,1
3518,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2,1
3519,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,2,1


In [ ]:
model = LinearRegression().fit(X,y)
coefs = dict(zip(X.columns, model.coef_))
print(coefs["lap_number"], coefs["tyre_age"], model.score(X, y))

-0.05067822179403747 0.037916742522311364 0.7673511613103566
{'driver_1': np.float64(-1.1226912444089834), 'driver_3': np.float64(-1.1132079478989858), 'driver_5': np.float64(-0.16854981304295605), 'driver_6': np.float64(-0.5296944146190317), 'driver_10': np.float64(-0.022120382461629787), 'driver_11': np.float64(2.693224578982387), 'driver_12': np.float64(-1.994880048589049), 'driver_14': np.float64(1.417650661016901), 'driver_16': np.float64(-1.6056887436080458), 'driver_18': np.float64(1.8841328988744228), 'driver_23': np.float64(0.5570533317170703), 'driver_30': np.float64(0.29614377508992834), 'driver_31': np.float64(0.05563589898957274), 'driver_41': np.float64(-0.03226618904932331), 'driver_43': np.float64(0.4158585671635246), 'driver_44': np.float64(-1.6676284124020073), 'driver_55': np.float64(0.7077283766300847), 'driver_63': np.float64(-1.8986555587931326), 'driver_77': np.float64(2.4206038341819838), 'driver_87': np.float64(-0.2926491677727248), 'lap_number': np.float64(-0.

In [7]:
def fit_race_correction(g):

    X = pd.get_dummies(g["driver_number"], prefix = "driver", dtype = float)
    X["lap_number"] = g["lap_number"]
    X["tyre_age"] = g["tyre_age"]
    y = g["lap_duration"]

    model = LinearRegression().fit(X,y)
    coefs = dict(zip(X.columns, model.coef_))

    return {
        "location": g["location"].iloc[0],
        "fuel": coefs["lap_number"],
        "deg": coefs["tyre_age"],
        "r2": model.score(X,y),
    }

In [8]:
g = races[races["location"] == "Melbourne"]
fit_race_correction(g)

{'location': 'Melbourne',
 'fuel': np.float64(-0.05067822179403747),
 'deg': np.float64(0.037916742522311364),
 'r2': 0.7673511613103566}

In [10]:
results = []

for _,g in races.groupby("session_key"):
    results.append(fit_race_correction(g))

results = pd.DataFrame(results)
print(results)


             location      fuel       deg        r2
0           Melbourne -0.050678  0.037917  0.767351
1            Shanghai -0.059084  0.020191  0.827675
2              Suzuka -0.071541  0.035022  0.836185
3       Miami Gardens -0.029846  0.035105  0.784132
4            Montréal -0.033923 -0.022195  0.731443
5         Monte Carlo -0.012454  0.025703  0.137222
6           Barcelona -0.059258  0.129425  0.820028
7           Spielberg -0.037714  0.076543  0.833093
8         Silverstone -0.060449  0.056132  0.896356
9   Spa-Francorchamps -0.061075  0.050215  0.862867
10           Budapest -0.049709  0.065859  0.802476
11          Zandvoort -0.044764  0.042657  0.728800
12              Monza -0.056201  0.013619  0.511651
13             Madrid -0.078584  0.004827  0.788295
14               Baku -0.067945 -0.005665  0.821312
15         Marina Bay -0.043164  0.023615  0.720159
16             Austin -0.063526  0.060091  0.850446
17        Mexico City -0.028146  0.029505  0.786282
18          

In [11]:
results[["fuel", "deg"]].median()

fuel   -0.056201
deg     0.029505
dtype: float64

## Race correction: fuel and tyre effects

**Model.** For each 2026 race I fitted a linear regression:
lap_time = driver_base + fuel × lap_number + deg × tyre_age.
Driver dummies (one-hot) absorb the pace difference between cars, so the
two remaining coefficients only measure fuel burn and tyre wear.

**Why races and not FP2.** In an FP2 stint, lap_number and tyre_age grow
together (+1 every lap), so the regression cannot separate them
(collinearity). In a race, tyre_age can be reset after a pit stop.

**Results (median over 21 races).**
- fuel = -0.056 s/lap: means that the pilot is faster every lap with 0.056s.
- deg  = +0.030 s/lap of tyre age: means that the pilot isd slower every lap with 0.030s.
- I use the median, not the mean, because a strange race should not have a huge impact. 